# Final Model Training

Model development and model selection were completed using temporal
cross-validation on the 2015–2022 training period.

The selected configurations were subsequently evaluated on the external
2023 validation set.

At this stage, all model choices are fixed. Each of the six selected models
is therefore trained one final time using all available pre-test data:

- Final training period: 2015–2023
- Final test period: 2024–2025

The 2024–2025 test set is used only once for final evaluation. It is not used
for hyperparameter selection, preprocessing fitting, feature-importance
analysis, or any other model-development decision.

This notebook trains the final models, generates final test predictions and
probabilities, evaluates their performance, and saves all outputs for the
final evaluation notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys
import joblib

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.neighbors import (
    KNeighborsClassifier
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Activation
)

from tensorflow.keras import Input
from tensorflow.keras import optimizers

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

from src.preprocessing import (
    fit_preprocessor,
    transform_data,
    numeric_features
)

from src.evaluation import (
    evaluate_model
)

In [2]:
#Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
#Paths
PROCESSED_DATA_DIR = Path(
    "../data/processed"
)

RESULTS_DIR = Path(
    "../results"
)

FINAL_RESULTS_DIR = (
    RESULTS_DIR
    / "final"
)

MODELS_DIR = Path(
    "../models"
)


FINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
#Load the prepared dataset
matches = pd.read_csv(
    PROCESSED_DATA_DIR
    / "feature_split_matches.csv",
    parse_dates=["Date"]
)


matches = (
    matches
    .sort_values(
        [
            "Date",
            "MatchID"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Dataset shape:",
    matches.shape
)

print()

print(
    matches[
        "DataSplit"
    ].value_counts()
)

Dataset shape: (26536, 53)

DataSplit
Training      18810
Test           5119
Validation     2607
Name: count, dtype: int64


In [5]:
#Construct final train/test split
final_train_data = (
    matches.loc[
        matches[
            "DataSplit"
        ].isin(
            [
                "Training",
                "Validation"
            ]
        )
    ]
    .copy()
)


test_data = (
    matches.loc[
        matches[
            "DataSplit"
        ]
        ==
        "Test"
    ]
    .copy()
)

print(
    "Final training matches:",
    len(final_train_data)
)

print(
    "Final test matches:",
    len(test_data)
)

print()

print(
    "Final training period:",
    final_train_data["Date"].min(),
    "-",
    final_train_data["Date"].max()
)

print(
    "Final test period:",
    test_data["Date"].min(),
    "-",
    test_data["Date"].max()
)

Final training matches: 21417
Final test matches: 5119

Final training period: 2015-01-05 00:00:00 - 2023-12-31 00:00:00
Final test period: 2024-01-01 00:00:00 - 2025-11-16 00:00:00


In [6]:
#Safety checks
assert (
    final_train_data[
        "Date"
    ].max()
    <
    test_data[
        "Date"
    ].min()
)


assert set(
    final_train_data[
        "DataSplit"
    ].unique()
) == {
    "Training",
    "Validation"
}


assert set(
    test_data[
        "DataSplit"
    ].unique()
) == {
    "Test"
}


print(
    "Final chronological split is correct."
)

Final chronological split is correct.


In [7]:
#Selected configurations
selected_configurations = pd.DataFrame(
    [
        {
            "Model":
                "ANN",
            "Configuration":
                "16 neurons, ReLU, learning rate 0.001, 50 epochs"
        },

        {
            "Model":
                "Random Forest",
            "Configuration":
                "200 trees, max depth 10"
        },

        {
            "Model":
                "k-NN",
            "Configuration":
                "151 neighbors, uniform weights"
        },

        {
            "Model":
                "AdaBoost",
            "Configuration":
                "4 max leaf nodes, 100 estimators, learning rate 0.5"
        },

        {
            "Model":
                "XGBoost",
            "Configuration":
                "max depth 2, 200 estimators, learning rate 0.05"
        },

        {
            "Model":
                "LightGBM",
            "Configuration":
                "7 leaves, 200 estimators, learning rate 0.05"
        }
    ]
)


selected_configurations

,Model,Configuration
0,ANN,"16 neurons, ReLU, learning rate 0.001, 50 epochs"
1,Random Forest,"200 trees, max depth 10"
2,k-NN,"151 neighbors, uniform weights"
3,AdaBoost,"4 max leaf nodes, 100 estimators, learning rat..."
4,XGBoost,"max depth 2, 200 estimators, learning rate 0.05"
5,LightGBM,"7 leaves, 200 estimators, learning rate 0.05"


In [8]:
#Final preprocessing
(
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
) = fit_preprocessor(
    final_train_data
)

X_final_train = transform_data(
    final_train_data,
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
)


X_test = transform_data(
    test_data,
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
)

y_final_train = (
    final_train_data[
        "Player1Won"
    ]
    .to_numpy()
)


y_test = (
    test_data[
        "Player1Won"
    ]
    .to_numpy()
)

In [9]:
#Check preprocessing
print(
    "Final training matrix:",
    X_final_train.shape
)

print(
    "Test matrix:",
    X_test.shape
)


assert (
    X_final_train.shape[1]
    ==
    X_test.shape[1]
)


assert (
    X_final_train
    .isna()
    .sum()
    .sum()
    ==
    0
)


assert (
    X_test
    .isna()
    .sum()
    .sum()
    ==
    0
)


print(
    "Final preprocessing check passed."
)

Final training matrix: (21417, 50)
Test matrix: (5119, 50)
Final preprocessing check passed.


In [10]:
#Save final preprocessing
final_preprocessing = {
    "numeric_medians":
        final_numeric_medians,

    "categorical_modes":
        final_categorical_modes,

    "one_hot_encoder":
        final_one_hot_encoder
}


joblib.dump(
    final_preprocessing,
    MODELS_DIR
    / "final_preprocessing.joblib"
)


print(
    "Final preprocessing saved."
)

Final preprocessing saved.


In [11]:
#Final scaler for ANN and k-NN
final_scaler = StandardScaler()


X_final_train_scaled = (
    X_final_train.copy()
)

X_test_scaled = (
    X_test.copy()
)

X_final_train_scaled[
    numeric_features
] = (
    final_scaler.fit_transform(
        X_final_train_scaled[
            numeric_features
        ]
    )
)


X_test_scaled[
    numeric_features
] = (
    final_scaler.transform(
        X_test_scaled[
            numeric_features
        ]
    )
)

joblib.dump(
    final_scaler,
    MODELS_DIR
    / "final_scaler.joblib"
)


print(
    "Final scaler saved."
)

Final scaler saved.


In [12]:
#ANN builder
def build_ann(
    input_features,
    hidden_layers,
    hidden_activation,
    learning_rate
):

    model = Sequential()

    model.add(
        Input(
            shape=(input_features,)
        )
    )

    for number_of_neurons in hidden_layers:

        model.add(
            Dense(
                number_of_neurons
            )
        )

        model.add(
            Activation(
                hidden_activation
            )
        )

    model.add(
        Dense(1)
    )

    model.add(
        Activation(
            "sigmoid"
        )
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    return model

In [13]:
#Final ANN
np.random.seed(42)
tf.random.set_seed(42)


X_ann_train = (
    X_final_train_scaled
    .to_numpy(
        dtype=np.float32
    )
)

X_ann_test = (
    X_test_scaled
    .to_numpy(
        dtype=np.float32
    )
)


final_ann = build_ann(
    input_features=
        X_ann_train.shape[1],

    hidden_layers=[
        16
    ],

    hidden_activation=
        "relu",

    learning_rate=
        0.001
)


final_ann.fit(
    X_ann_train,
    y_final_train,

    epochs=50,
    batch_size=128,

    verbose=1
)

Epoch 1/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6350 - loss: 0.6393  
Epoch 2/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6534 - loss: 0.6168
Epoch 3/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6565 - loss: 0.6123  
Epoch 4/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6590 - loss: 0.6099
Epoch 5/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6592 - loss: 0.6083
Epoch 6/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6601 - loss: 0.6072
Epoch 7/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6613 - loss: 0.6064
Epoch 8/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6623 - loss: 0.6057
Epoch 9/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6633 - loss: 0.6050
Epoch 10/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6636 - loss: 0.6045
Epoch 11/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6645 - loss: 0.6040
Epoch 12/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/

In [14]:
#ANN final predictions
ann_test_probabilities = (
    final_ann.predict(
        X_ann_test,
        verbose=0
    )
    .reshape(-1)
)


ann_test_predictions = (
    ann_test_probabilities
    >=
    0.5
).astype(int)


ann_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            ann_test_predictions,

        model_name=
            "ANN",

        y_probability=
            ann_test_probabilities
    )
)


ann_test_result

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,ANN,0.639578,0.639702,0.647927,0.628141,0.63788,0.704121,0.623636


In [15]:
#Save ANN
final_ann.save(
    MODELS_DIR
    / "final_ann.keras"
)


print(
    "Final ANN saved."
)

Final ANN saved.


In [16]:
#Final Random Forest
final_rf = RandomForestClassifier(
    n_estimators=200,

    max_depth=10,

    min_samples_leaf=1,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)


final_rf.fit(
    X_final_train,
    y_final_train
)

rf_test_predictions = (
    final_rf.predict(
        X_test
    )
)


rf_test_probabilities = (
    final_rf.predict_proba(
        X_test
    )[:, 1]
)


rf_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            rf_test_predictions,

        model_name=
            "Random Forest",

        y_probability=
            rf_test_probabilities
    )
)


rf_test_result

joblib.dump(
    final_rf,
    MODELS_DIR
    / "final_rf.joblib"
)


print(
    "Final Random Forest saved."
)

Final Random Forest saved.


In [17]:
#Final k-NN
final_knn = KNeighborsClassifier(
    n_neighbors=151,

    weights="uniform",

    n_jobs=-1
)


final_knn.fit(
    X_final_train_scaled,
    y_final_train
)

knn_test_predictions = (
    final_knn.predict(
        X_test_scaled
    )
)


knn_test_probabilities = (
    final_knn.predict_proba(
        X_test_scaled
    )[:, 1]
)


knn_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            knn_test_predictions,

        model_name=
            "k-NN",

        y_probability=
            knn_test_probabilities
    )
)


knn_test_result

joblib.dump(
    final_knn,
    MODELS_DIR
    / "final_knn.joblib"
)


print(
    "Final k-NN saved."
)

Final k-NN saved.


In [18]:
#Final AdaBoost
final_weak_tree = (
    DecisionTreeClassifier(
        max_leaf_nodes=4,

        random_state=42
    )
)


final_adaboost = (
    AdaBoostClassifier(
        estimator=
            final_weak_tree,

        n_estimators=100,

        learning_rate=0.5,

        random_state=42
    )
)


final_adaboost.fit(
    X_final_train,
    y_final_train
)

adaboost_test_predictions = (
    final_adaboost.predict(
        X_test
    )
)


adaboost_test_probabilities = (
    final_adaboost.predict_proba(
        X_test
    )[:, 1]
)


adaboost_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            adaboost_test_predictions,

        model_name=
            "AdaBoost",

        y_probability=
            adaboost_test_probabilities
    )
)


adaboost_test_result

joblib.dump(
    final_adaboost,
    MODELS_DIR
    / "final_adaboost.joblib"
)


print(
    "Final AdaBoost saved."
)

Final AdaBoost saved.


In [19]:
#Final XGBoost
final_xgboost = XGBClassifier(
    n_estimators=200,

    max_depth=2,

    learning_rate=0.05,

    objective=
        "binary:logistic",

    eval_metric=
        "logloss",

    importance_type=
        "gain",

    random_state=42,

    n_jobs=-1
)


final_xgboost.fit(
    X_final_train,
    y_final_train
)

xgboost_test_predictions = (
    final_xgboost.predict(
        X_test
    )
)


xgboost_test_probabilities = (
    final_xgboost.predict_proba(
        X_test
    )[:, 1]
)


xgboost_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            xgboost_test_predictions,

        model_name=
            "XGBoost",

        y_probability=
            xgboost_test_probabilities
    )
)


xgboost_test_result

joblib.dump(
    final_xgboost,
    MODELS_DIR
    / "final_xgboost.joblib"
)


print(
    "Final XGBoost saved."
)

Final XGBoost saved.


In [20]:
#Final LightGBM
final_lightgbm = LGBMClassifier(
    num_leaves=7,

    n_estimators=200,

    learning_rate=0.05,

    objective="binary",

    random_state=42,

    n_jobs=-1,

    verbosity=-1
)


final_lightgbm.fit(
    X_final_train,
    y_final_train
)

lightgbm_test_predictions = (
    final_lightgbm.predict(
        X_test
    )
)


lightgbm_test_probabilities = (
    final_lightgbm.predict_proba(
        X_test
    )[:, 1]
)


lightgbm_test_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            lightgbm_test_predictions,

        model_name=
            "LightGBM",

        y_probability=
            lightgbm_test_probabilities
    )
)


lightgbm_test_result

joblib.dump(
    final_lightgbm,
    MODELS_DIR
    / "final_lightgbm.joblib"
)


print(
    "Final LightGBM saved."
)

Final LightGBM saved.


In [21]:
#Combine the six final results
final_model_results = pd.concat(
    [
        ann_test_result,
        rf_test_result,
        knn_test_result,
        adaboost_test_result,
        xgboost_test_result,
        lightgbm_test_result
    ],

    ignore_index=True
)

#Sort them
final_model_results = (
    final_model_results
    .sort_values(
        by=[
            "Accuracy",
            "ROC-AUC"
        ],

        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


final_model_results

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
5,ANN,0.639578,0.639702,0.647927,0.628141,0.637880,0.704121,0.623636


In [22]:
#Save final model metrics
final_model_results.to_csv(
    FINAL_RESULTS_DIR
    / "final_model_results.csv",

    index=False
)


print(
    "Final model metrics saved."
)

Final model metrics saved.


In [23]:
#Better-ranked-player baseline
assert (
    test_data[
        [
            "Player1Rank",
            "Player2Rank"
        ]
    ]
    .isna()
    .sum()
    .sum()
    ==
    0
)

ranking_baseline_predictions = (
    test_data[
        "Player1Rank"
    ]
    <
    test_data[
        "Player2Rank"
    ]
).astype(int).to_numpy()

ranking_baseline_result = (
    evaluate_model(
        y_true=
            y_test,

        y_pred=
            ranking_baseline_predictions,

        model_name=
            "Better-ranked player"
    )
)


ranking_baseline_result

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1
0,Better-ranked player,0.642508,0.642467,0.646308,0.646308,0.646308


In [24]:
#Models vs baseline
final_test_comparison = (
    pd.concat(
        [
            final_model_results,
            ranking_baseline_result
        ],

        ignore_index=True
    )
)

final_test_comparison = (
    final_test_comparison
    .sort_values(
        "Accuracy",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


final_test_comparison

final_test_comparison.to_csv(
    FINAL_RESULTS_DIR
    / "final_test_comparison.csv",

    index=False
)

In [25]:
#Create ONE unified final prediction file
analysis_columns = [
    "MatchID",
    "Date",
    "Tournament",
    "Surface",
    "Round",

    "Player1",
    "Player2",

    "Player1Rank",
    "Player2Rank",

    "RankDifference",
    "PointsDifference",

    "WinRateDifference",
    "Recent5WinRateDifference",
    "SurfaceWinRateDifference",

    "H2HMatchesBefore",

    "Player1Won"
]

final_test_predictions = (
    test_data[
        analysis_columns
    ]
    .copy()
)

In [26]:
#Add ANN outputs
final_test_predictions[
    "ANNPrediction"
] = ann_test_predictions


final_test_predictions[
    "ANNProbability"
] = ann_test_probabilities


final_test_predictions[
    "ANNCorrect"
] = (
    ann_test_predictions
    ==
    y_test
)


final_test_predictions[
    "ANNConfidence"
] = np.maximum(
    ann_test_probabilities,

    1
    -
    ann_test_probabilities
)

In [27]:
#Add RF outputs
final_test_predictions[
    "RFPrediction"
] = rf_test_predictions


final_test_predictions[
    "RFProbability"
] = rf_test_probabilities


final_test_predictions[
    "RFCorrect"
] = (
    rf_test_predictions
    ==
    y_test
)


final_test_predictions[
    "RFConfidence"
] = np.maximum(
    rf_test_probabilities,

    1
    -
    rf_test_probabilities
)

In [28]:
#Add k-NN outputs
final_test_predictions[
    "kNNPrediction"
] = knn_test_predictions


final_test_predictions[
    "kNNProbability"
] = knn_test_probabilities


final_test_predictions[
    "kNNCorrect"
] = (
    knn_test_predictions
    ==
    y_test
)


final_test_predictions[
    "kNNConfidence"
] = np.maximum(
    knn_test_probabilities,

    1
    -
    knn_test_probabilities
)

In [30]:
#Add AdaBoost outputs
final_test_predictions[
    "AdaBoostPrediction"
] = adaboost_test_predictions


final_test_predictions[
    "AdaBoostProbability"
] = adaboost_test_probabilities


final_test_predictions[
    "AdaBoostCorrect"
] = (
    adaboost_test_predictions
    ==
    y_test
)


final_test_predictions[
    "AdaBoostConfidence"
] = np.maximum(
    adaboost_test_probabilities,

    1
    -
    adaboost_test_probabilities
)

In [31]:
#Add XGBoost outputs
final_test_predictions[
    "XGBoostPrediction"
] = xgboost_test_predictions


final_test_predictions[
    "XGBoostProbability"
] = xgboost_test_probabilities


final_test_predictions[
    "XGBoostCorrect"
] = (
    xgboost_test_predictions
    ==
    y_test
)


final_test_predictions[
    "XGBoostConfidence"
] = np.maximum(
    xgboost_test_probabilities,

    1
    -
    xgboost_test_probabilities
)

In [32]:
#Add LightGBM outputs
final_test_predictions[
    "LightGBMPrediction"
] = lightgbm_test_predictions


final_test_predictions[
    "LightGBMProbability"
] = lightgbm_test_probabilities


final_test_predictions[
    "LightGBMCorrect"
] = (
    lightgbm_test_predictions
    ==
    y_test
)


final_test_predictions[
    "LightGBMConfidence"
] = np.maximum(
    lightgbm_test_probabilities,

    1
    -
    lightgbm_test_probabilities
)

In [33]:
#Add baseline prediction
final_test_predictions[
    "RankingBaselinePrediction"
] = (
    ranking_baseline_predictions
)


final_test_predictions[
    "RankingBaselineCorrect"
] = (
    ranking_baseline_predictions
    ==
    y_test
)

In [34]:
#Save unified predictions
final_test_predictions.to_csv(
    FINAL_RESULTS_DIR
    / "final_test_predictions.csv",

    index=False
)


print(
    "Final test predictions saved."
)

Final test predictions saved.


In [35]:
#Verify saved outputs
print(
    "Final result files:"
)

for file_path in sorted(
    FINAL_RESULTS_DIR.glob(
        "*.csv"
    )
):
    
    print(
        "-",
        file_path.name
    )

Final result files:
- final_model_results.csv
- final_test_comparison.csv
- final_test_predictions.csv


In [36]:
#Sanity check predictions
assert len(
    final_test_predictions
) == len(
    test_data
)


assert (
    final_test_predictions[
        [
            "ANNProbability",
            "RFProbability",
            "kNNProbability",
            "AdaBoostProbability",
            "XGBoostProbability",
            "LightGBMProbability"
        ]
    ]
    .isna()
    .sum()
    .sum()
    ==
    0
)


print(
    "Final prediction checks passed."
)

final_test_predictions.head()

Final prediction checks passed.


,MatchID,Date,Tournament,Surface,Round,Player1,Player2,Player1Rank,Player2Rank,RankDifference,...,XGBoostPrediction,XGBoostProbability,XGBoostCorrect,XGBoostConfidence,LightGBMPrediction,LightGBMProbability,LightGBMCorrect,LightGBMConfidence,RankingBaselinePrediction,RankingBaselineCorrect
21417,21417,2024-01-01,Brisbane International,Hard,1st Round,Murray A.,Dimitrov G.,42.0,14.0,-28.0,...,0,0.383699,True,0.616301,0,0.316778,True,0.683222,0,True
21418,21418,2024-01-01,Brisbane International,Hard,1st Round,Rune H.,Purcell M.,8.0,45.0,37.0,...,1,0.778013,True,0.778013,1,0.775602,True,0.775602,1,True
21419,21419,2024-01-01,Brisbane International,Hard,1st Round,Shelton B.,Safiullin R.,17.0,39.0,22.0,...,1,0.634187,False,0.634187,1,0.643646,False,0.643646,1,False
21420,21420,2024-01-01,Hong Kong Tennis Open,Hard,1st Round,Borges N.,Kotov P.,66.0,67.0,1.0,...,0,0.440316,True,0.559684,0,0.438579,True,0.561421,1,False
21421,21421,2024-01-01,Hong Kong Tennis Open,Hard,1st Round,Bonzi B.,Ruusuvuori E.,73.0,69.0,-4.0,...,0,0.485134,True,0.514866,1,0.525389,False,0.525389,0,True


In [37]:
#Final summary
print(
    "=" * 70
)

print(
    "FINAL MODEL TRAINING COMPLETE"
)

print(
    "=" * 70
)


print(
    "\nFinal training period:"
)

print(
    final_train_data[
        "Date"
    ].min(),
    "-",
    final_train_data[
        "Date"
    ].max()
)


print(
    "\nFinal test period:"
)

print(
    test_data[
        "Date"
    ].min(),
    "-",
    test_data[
        "Date"
    ].max()
)


print(
    "\nFinal model results:"
)


display(
    final_model_results
)


print(
    "\nModels + baseline:"
)


display(
    final_test_comparison
)

FINAL MODEL TRAINING COMPLETE

Final training period:
2015-01-05 00:00:00 - 2023-12-31 00:00:00

Final test period:
2024-01-01 00:00:00 - 2025-11-16 00:00:00

Final model results:


,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
5,ANN,0.639578,0.639702,0.647927,0.628141,0.637880,0.704121,0.623636



Models + baseline:


,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,Better-ranked player,0.642508,0.642467,0.646308,0.646308,0.646308,NaN,NaN
5,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
6,ANN,0.639578,0.639702,0.647927,0.628141,0.637880,0.704121,0.623636
